<a href="https://colab.research.google.com/github/zhangyuwen193-cell/114-2-Programing-Language/blob/main/hw2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [ ]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data, then applies them to make predictions, decisions, or create new things.


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Lwzfk3Cl_XTiQ6IV406924NtWoCZ1slo-k4fcMLBJFE/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [ ]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [ ]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：98
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 98

請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：85
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 85

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：50
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 50

請輸入科目（或輸入 'q' 停止）：q


In [ ]:
new_grades

[['2026-03-26', '數學', 98], ['2026-03-26', '國文', 85], ['2026-03-26', '數學', 50]]

In [ ]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的學生單日成績所做的簡單摘要與常見迷思整理，其中不包含任何評分或評價。\n\n---\n\n### 成績摘要\n\n這份成績紀錄了學生在 **2026年3月26日** 的表現，共包含三筆成績：\n\n*   **數學：98分**\n*   **國文：85分**\n*   **數學：50分**\n\n值得注意的是，數學科目有兩筆成績，分別為98分和50分；國文科目則有一筆成績為85分。\n\n---\n\n### 常見迷思整理\n\n在解讀這類成績列表時，人們常會產生一些基於單純數字的迷思，但這些迷思往往忽略了成績背後的重要情境。以下是一些常見的迷思及其澄清：\n\n1.  **迷思一：單一分數能全面評斷學習能力**\n    *   **澄清：** 學生在同一科目（數學）有兩筆差異較大的成績（98分與50分），這清楚表明單一的成績往往只反映了特定測驗、作業或評量方式的表現，不能完整代表學生的整體學習能力或潛力。了解分數背後的測驗類型、內容難度、範圍以及學生當時的準備情況更為重要。\n\n2.  **迷思二：不同科目成績可直接比較優劣**\n    *   **澄清：** 國文85分與數學的兩筆成績（98分、50分）性質不同。不同科目間的成績受教學方式、評量標準、考試題型、個人興趣特長、甚至考試當天的狀態等因素影響，難以直接橫向比較其學習優劣。每個科目都有其獨特的學習目標與評量方式。\n\n3.  **迷思三：成績是衡量學習成果的唯一標準**\n    *   **澄清：** 雖然成績提供量化的回饋，但它只是學習成果的一個面向。學生的學習過程、努力程度、面對挑戰的態度、解決問題的能力、批判性思維、創造力以及課堂參與等，都是重要且無法完全透過分數體現的學習價值。\n\n4.  **迷思四：高分代表無須改進，低分代表無望**\n    *   **澄清：** 98分顯示了在該次數學評量中優秀的掌握度，但仍可能在細節或更深層次有進步空間。50分則明確指出有需要關注和加強的特定領域，而非能力的終極判斷。每個分數都是學習改進的起點，提供學生和老師反思與調整策略的機會。\n\n5.  **迷思五：所有成績都代表相同類型的評量**\n    *   **澄清：** 列表中的三筆成績，特別是同一科目有不同分數，可能代表了不同形式的評量（例如，一次是單元測驗，

In [ ]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：50
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 50

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：60
已記錄：日期: 2026-03-26, 科目: 英文, 成績: 60

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：80
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 80

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程不帶任何評價，僅做客觀陳述。

---

### 學生成績摘要與常見迷思整理

**日期：** 2026年3月26日

---

#### 成績摘要

*   **科目與分數：**
    *   國文：50分
    *   英文：60分
    *   數學：80分
*   **分數分佈：** 學生在這次三個科目中的分數範圍介於50分至80分之間。
*   **科目表現（事實描述）：** 數學取得最高分數（80分），英文次之（60分），國文分數為50分。
*   **平均分數：** 這三科的平均分數約為63.33分。

---

#### 常見迷思整理（關於成績，非針對特定學生）

在審視學術成績時，人們常會有一些既定觀念，以下列出幾個常見的迷思及其更全面的視角：

1.  **迷思一：單次成績能全面評估學習能力。**
    *   **更全面的視角：** 成績是特定時間點、在特定測驗形式下的一種表現。它受考試形式、題目難度、學生當日狀態、學習策略等多種因素影響，不一定能完全反映學生的學習潛力、努力過程、對知識的實際理解深度或非學術能力（如批判性思維、解決問題、創造力等）。

2.  **迷思二：分數高低直接決定未來成功。**